# Piano B — Notebook F: robustezza alla ricetta (CBraMod a due fasi) e pretraining su clip non viste

Sessione 02, Parti A e B. In ordine:
1. Optuna `cbramod_twophase` (30 trial, validation sub080-099): testa (appiattita ufficiale / pooling per
   canale / pooling globale), standardizzazione per soggetto degli embedding, epoche e lr delle due fasi.
2. Controllo di sanità: CBraMod a due fasi su split FM, 1 seed, BA di validation > 0.20. Se fallisce, i job
   della Parte A non partono: fermarsi e diagnosticare.
3. Parte A: pre-addestrato e da zero, split FM e leave-clip-out (3 seed), 2 modelli permutati per fold.
4. Parte B: da zero con warm-up e pre-addestrato con warm-up sul leave-clip-out (3 seed).

L'ablazione A.4 (coda `F_bis`) è solo preparata: si lancia solo se il CBraMod a due fasi supera il caso sulle
clip non viste (tools/aggregate.py stampa la decisione).

Impostazioni: Accelerator **GPU T4 x2**, Internet off. Input: `pianob-faced-data`, `pianob-code`
(+ l'output della sessione precedente di questo notebook, per riprendere).

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
ACCOUNT = "nbF_ricetta_pretraining"
os.environ["PIANOB_REQUIRE_HPO"] = "0" if SYNTHETIC else "1"   # lo studio Optuna della coda è escluso dal controllo
nb_helpers.weights_check()
nb_helpers.run_queue("F", gpus="auto", synthetic=SYNTHETIC)
nb_helpers.queue_status("F")

In [ ]:
nb_helpers.finish(ACCOUNT)